In [3]:
import paho.mqtt.client as mqtt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import time
from datetime import datetime
import os

# ─── Config ────────────────────────────────────────────────
BROKER_ADDRESS = "127.0.0.1"
MQTT_TOPIC = "imu/data"
RECORD_SECONDS = 10
OUTPUT_DIR = "tc01_sampling_test"

os.makedirs(OUTPUT_DIR, exist_ok=True)

timestamps = []
collecting = True

# ─── MQTT Callbacks ─────────────────────────────────────────
def on_connect(client, userdata, flags, rc):
    if rc == 0:
        print("✅ Connected to MQTT Broker")
        client.subscribe(MQTT_TOPIC)
    else:
        print(f"❌ Failed with code {rc}")

def on_message(client, userdata, msg):
    global timestamps, collecting
    if not collecting:
        return

    pkt = msg.payload.decode("utf-8").strip()

    # Extract the content inside [[ ... ]]
    match = re.findall(r"\[([0-9\.\,\-\s]+)\]", pkt)
    if not match:
        return

    # Convert the comma-separated values into floats
    values = [float(v) for v in match[0].split(",")]

    # Expecting: timestamp, seq, ax, ay, az, gx, gy, gz → 8 values
    if len(values) == 8:
        ts = int(values[0])   # first value = timestamp
        timestamps.append(ts)

    print(f"📡 Packets received: {len(timestamps)}", end="\r")

# ─── Record for N seconds ───────────────────────────────────
client = mqtt.Client()
client.on_connect = on_connect
client.on_message = on_message

print(f"Starting {RECORD_SECONDS}s packet collection...")
client.connect(BROKER_ADDRESS, 1883, 60)
client.loop_start()

start = time.time()
while time.time() - start < RECORD_SECONDS:
    time.sleep(0.02)

collecting = False
client.loop_stop()
client.disconnect()

print(f"\nFinished. Total packets: {len(timestamps)}")

# ─── Data Preparation ──────────────────────────────────────
if len(timestamps) == 0:
    print("⚠️ No data received. Check MQTT topic / Arduino sketch.")
else:
    df = pd.DataFrame(timestamps, columns=["timestamp"])

    # Compute ∆t between consecutive packets
    df["dt"] = df["timestamp"].diff()

    # Handle micros() wrap-around
    df.loc[df["dt"] < 0, "dt"] += 2**32 

    df["dt_sec"] = df["dt"] / 1_000_000.0

    # Stats
    mean_dt_ms = df["dt_sec"].mean() * 1000
    std_dt_ms = df["dt_sec"].std() * 1000
    freq_hz = 1.0 / df["dt_sec"].mean()

    print("\n──── MQTT Sampling Rate Summary ────")
    print(f"Packets received: {len(df)}")
    print(f"Average frequency: {freq_hz:.2f} Hz")
    print(f"Mean interval: {mean_dt_ms:.3f} ms")
    print(f"Jitter (std dev): {std_dt_ms:.3f} ms")

    # ─── Save CSV ───────────────────────────────────────────
    date_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"tc01_mqtt_rate_{date_str}_{len(df)}pkts_{RECORD_SECONDS}s.csv"
    filepath = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(filepath, index=False)

    print(f"\n💾 Saved CSV: {filepath}")




/var/folders/m1/qxpcclg55hg_8gcmt4vpkrq80000gn/T/ipykernel_78463/4084063274.py:52: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client()


Starting 10s packet collection...
✅ Connected to MQTT Broker


📡 Packets received: 265
Finished. Total packets: 265

──── MQTT Sampling Rate Summary ────
Packets received: 265
Average frequency: 26.56 Hz
Mean interval: 37.645 ms
Jitter (std dev): 0.495 ms

💾 Saved CSV: tc01_sampling_test/tc01_mqtt_rate_20251120_184610_265pkts_10s.csv
📡 Packets received: 265
Finished. Total packets: 265

──── MQTT Sampling Rate Summary ────
Packets received: 265
Average frequency: 26.56 Hz
Mean interval: 37.645 ms
Jitter (std dev): 0.495 ms

💾 Saved CSV: tc01_sampling_test/tc01_mqtt_rate_20251120_184610_265pkts_10s.csv
